In [ ]:
%py
# PySpark Test Code for Databricks Environment
# This script tests various features required for managing inventory transactions including appl_qty calculation

from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import SparkSession
from chispa.dataframe_comparer import *
import unittest

# Initialize the Spark session if not already available in the environment
spark = SparkSession.builder.appName("Databricks_Test").getOrCreate()

# Define sample schema for test DataFrame
schema = StructType([
    StructField("txn_id", StringType(), False),
    StructField("ref_txn_qty", DecimalType(3,1), False),
    StructField("cumulative_txn_qty", DecimalType(4,1), False),
    StructField("cumulative_ref_ord_sched_qty", DecimalType(4,1), False),
    StructField("ref_ord_sched_qty", DecimalType(3,1), False),
    StructField("prior_cumulative_txn_qty", DecimalType(3,1), False),
    StructField("prior_cumulative_ref_ord_sched_qty", DecimalType(3,1), False),
    StructField("apl_qty", DecimalType(5,1), True)
])

# Sample data for testing purpose
data = [
    ("1", 50.0, 100.0, 90.0, 50.0, 40.0, 30.0, 40.0),
    ("2", -10.0, 80.0, 70.0, 40.0, 50.0, 45.0, -10.0),
    ("3", 20.0, 60.0, 100.0, 30.0, 30.0, 25.0, 30.0),
    ("4", 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, None),
    ("5", 999.9, 9999.0, 9999.0, 999.9, 999.0, 999.0, 999.9),
    # Include invalid data format case
    ("6", None, 100.0, 90.0, 50.0, 40.0, 30.0, None),
    ("7", 55.5, 110.0, 100.0, 55.5, 45.0, 35.0, 45.5)
]

# Create DataFrame using the sample data and schema
test_df = spark.createDataFrame(data, schema)

class TestInventoryTransactions(unittest.TestCase):

    def test_data_type_consistency(self):
        """
        Test to ensure correct data types and NULL handling
        """
        try:
            # Validate specific column data types
            self.assertEqual(test_df.schema["ref_txn_qty"].dataType, DecimalType(3,1))
            self.assertEqual(test_df.schema["apl_qty"].nullable, True)
        except Exception as e:
            self.fail(f"Schema validation failed with exception: {str(e)}")

    def test_apl_qty_calculation_logic(self):
        """
        Validate the logic for apl_qty calculation
        """
        try:
            # Define expected DataFrame with calculated apl_qty
            expected_data = [
                ("1", 50.0, 100.0, 90.0, 50.0, 40.0, 30.0, 40.0),
                ("2", -10.0, 80.0, 70.0, 40.0, 50.0, 45.0, -10.0),
                ("3", 20.0, 60.0, 100.0, 30.0, 30.0, 25.0, 30.0),
                ("4", 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, None),
                ("5", 999.9, 9999.0, 9999.0, 999.9, 999.0, 999.0, 999.9),
                ("7", 55.5, 110.0, 100.0, 55.5, 45.0, 35.0, 45.5)
            ]
            expected_df = spark.createDataFrame(expected_data, schema)
            
            # Assert: Compare expected and actual DataFrames
            assert_df_are_equal(expected_df,test_df.select(expected_df.columns))
        except AssertionError as e:
            self.fail(f"Apl_qty calculation logic validation failed: {str(e)}")

    def test_integration_functionality(self):
        """
        Test integration with external inventory and transactions
        """
        # Assuming integration logic here
        
        try:
            # Verify data synchronization between tables
            main_table_df = spark.table("purgo_playground.purgo_playground.f_inv_movmnt_apl_qty")
            self.assertEqual(test_df.count(), main_table_df.count())
        except Exception as e:
            self.fail(f"Data integration test failed: {str(e)}")

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

